In [ ]:
# 산탄데르 은행 고객 데이터로 만족 예측 모델 프로젝트
# 피쳐 : 370개
# 원래답 : 1(불만족), 0(만족)
# 이 예제로 무엇을 학습 목표
# 1. 불균형 데이터 처리
# 2. info(), describe() => 위장된 결측치
# 3. AUC
# 4. HyperOpt, Kfold
# 5. XGB, LGBM


In [ ]:
# 모델 파이프라인
# 1. 데이터로딩 2. 불균형확인 3. 이상값 탐지 4.전처리 5.3단계 분할 6. XGB 학습 7. HyperOpt 튜닝 8. LGBM

In [1]:
# numpy : 수치 연산(배열, 행렬 계산 등)을 위한 라이브러리 임포트
import numpy as np
# pandas : 데이터프레임 기반 데이터 처리/분석 라이브러리 임포트
import pandas as pd
# matplotlib.pyplot : 그래프/시각화를 위한 모듈 임포트
import matplotlib.pyplot as plt
# matplotlib : 시각화 설정(폰트 등)을 위해 matplotlib 자체도 임포트
import matplotlib
# warnings : 경고 메시지 제어를 위한 표준 라이브러리 임포트
import warnings
# warnings.filterwarnings('ignore') : 실행 중 발생하는 경고 메시지를 화면에 출력하지 않도록 설정
warnings.filterwarnings('ignore')


# 1. 데이터 로딩(5초)
# dataset shape: (76020 rows, 371 columns)

# pd.read_csv(경로, encoding='latin-1') : 지정 경로의 csv 파일을 읽어 데이터프레임으로 로드
#   - encoding='latin-1' : 파일 인코딩을 latin-1(서유럽어 인코딩)으로 지정하여 인코딩 오류 방지
cust_df = pd.read_csv("../data/santander-customer-satisfaction/train.csv", encoding='latin-1')

# cust_df.shape : (행 개수, 열 개수) 튜플 반환 → 로드된 데이터 크기 확인
print('dataset shape:', cust_df.shape)

# cust_df.head(3) : 데이터프레임의 상위 30개 행을 출력하여 데이터 내용을 미리 확인
cust_df.head(3)

dataset shape: (76020, 371)


,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,1,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39205.17,0
1,3,2,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49278.03,0
2,4,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67333.77,0


In [2]:
# var3 , id

# cust_df['var3'] : cust_df 데이터프레임에서 'var3' 컬럼(Series)을 선택

# .replace(-999999, 2) : 값 치환 함수
#   - 첫 번째 인자(-999999) : 치환 대상이 되는 원래 값
#   - 두 번째 인자(2)       : 치환할 새로운 값
#   - 'var3' 컬럼 내에서 이상치(결측 표시용 특수값)로 쓰인 -999999를 2로 변경
#   → -999999는 실제 값이 아니라 결측치를 나타내는 코드값으로 추정되며, 최빈값 등으로 대체하는 전처리 작업
cust_df['var3'] = cust_df['var3'].replace(-999999, 2)

In [3]:
# id 삭제, inplace=False로 해야 cust_df에 반영되지 않는다. 복사본으로 처리해라.
cust_df.drop('ID', axis=1, inplace=True)

In [9]:
# 데이터와 레이블 분리
# 데이터 컬럼전체(370개)에서 마지막 컬럼(TARGET)만 제외하고 피처로 사용

# cust_df.iloc[:, :-1] : cust_df의 모든 행(:)과, 마지막 열을 제외한 모든 열(:-1)을 선택
#   - iloc은 위치(정수 인덱스) 기반 선택자 → [행 범위, 열 범위]
#   - :-1은 '처음부터 마지막 바로 앞까지'를 의미 (마지막 컬럼인 TARGET 제외)
# X_features = cust_df

# 레이블은 마지막 컬럼
# cust_df.iloc[:, -1] : cust_df의 모든 행(:)과 마지막 열(-1) 하나만 선택 → Series로 반환
# y_labels   = cust_df.iloc[:, -1]

# X_features.shape : (행 개수, 열 개수) 튜플 반환 → 피처 데이터의 크기 확인
# y_labels.shape   : (행 개수,) 튜플 반환 → 레이블 데이터의 크기 확인
# f'...' : f-string, 문자열 내 {} 안에 변수 값을 바로 삽입해 출력
# print(f'피처 데이터 shape: {X_features.shape}, 레이블 shape: {y_labels.shape}')

피처 데이터 shape: (76020, 370), 레이블 shape: (76020,)


In [12]:
# ① Zero Variance 제거
variance_zero_cols = [col for col in cust_df.columns if cust_df[col].var() == 0]
cust_df.drop(columns=variance_zero_cols, inplace=True)
print(f"1. Zero Variance 제거 후 피처 수: {cust_df.shape[1]} (제거: {len(variance_zero_cols)}개)")

# ② 중복 피처 제거 (12개 중 정확히 절반인 6개 세트 제거 효과와 동일)
duplicated_cols = cust_df.T.duplicated()[cust_df.T.duplicated()].index.tolist()
cust_df.drop(columns=duplicated_cols, inplace=True)
print(f"2. 중복 피처 제거 후 피처 수: {cust_df.shape[1]} (제거: {len(duplicated_cols)}개)")

# ③ 희소 특징 제거 (99% 이상이 0으로 채워진 값)
# shape[0]으로 정확한 전체 행 수를 분모로 지정합니다.
num_rows = cust_df.shape[0]
sparse_cols = [col for col in cust_df.columns if (cust_df[col] == 0).sum() / num_rows >= 0.99]
cust_df.drop(columns=sparse_cols, inplace=True)
print(f"3. 희소 피처(99%가 0) 제거 후 피처 수: {cust_df.shape[1]} (제거: {len(sparse_cols)}개)")
print("-> 논문 플로우를 거쳐 최종 정제된 피처 개수 확보 완료\n")

cust_df.head()

1. Zero Variance 제거 후 피처 수: 144 (제거: 0개)
2. 중복 피처 제거 후 피처 수: 144 (제거: 0개)
3. 희소 피처(99%가 0) 제거 후 피처 수: 144 (제거: 0개)
-> 논문 플로우를 거쳐 최종 정제된 피처 개수 확보 완료



,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var41_comer_ult1,imp_op_var41_comer_ult3,imp_op_var41_efect_ult1,imp_op_var41_efect_ult3,imp_op_var41_ult1,...,saldo_medio_var12_hace2,saldo_medio_var12_hace3,saldo_medio_var12_ult1,saldo_medio_var12_ult3,saldo_medio_var13_corto_hace2,saldo_medio_var13_corto_hace3,saldo_medio_var13_corto_ult1,saldo_medio_var13_corto_ult3,var38,TARGET
0,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.00,0.00,0.0,0.00,0.0,0.00,39205.170000,0
1,2,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.00,0.00,300.0,122.22,300.0,240.75,49278.030000,0
2,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.00,0.00,0.0,0.00,0.0,0.00,67333.770000,0
3,2,37,0.0,195.0,195.0,195.0,195.0,0.0,0.0,195.0,...,0.0,0.0,0.00,0.00,0.0,0.00,0.0,0.00,64007.970000,0
4,2,39,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,85501.89,85501.89,0.0,0.00,0.0,0.00,117310.979016,0


In [15]:
# info() : 전체 데이터 구조파악
cust_df.head()

X_features = cust_df.iloc[:, :-1]

# 레이블은 마지막 컬럼
# cust_df.iloc[:, -1] : cust_df의 모든 행(:)과 마지막 열(-1) 하나만 선택 → Series로 반환
y_labels   = cust_df.iloc[:, -1]

In [20]:
# 학습/테스트 데이터 분리, 분포 확인

# sklearn에서 데이터를 학습/테스트용으로 무작위 분할해주는 함수를 가져옴
from sklearn.model_selection import train_test_split


# train_test_split() 파라미터 설명
#   - X_features : 분할할 입력(피처) 데이터
#   - y_labels   : 분할할 정답(레이블) 데이터 (X_features와 짝을 맞춰 함께 분할됨)
#   - test_size=0.2   : 전체 데이터 중 테스트 세트로 뗄 비율 (0.2 = 20%, 나머지 80%는 학습 세트)
#   - random_state=0  : 무작위 분할 시 시드값 고정. 값을 고정하면 코드를 몇 번 실행해도 동일한 결과로 분할됨
# 반환값 순서: X_train(학습용 입력), X_test(테스트용 입력), y_train(학습용 정답), y_test(테스트용 정답)
# 데이터를 분할하기 위한 train_test_split() 함수를 사용하여 학습용 데이터와 테스트용 데이터를 나눔
# 1. 입력데이터 / 2. 정답데이터 / 3. 테스트세트 비율 / 4. 시드값
# 원 데이터의 입력데이터와 정답데이의 테스트 비율값을 주고 
# random_state=0처럼 값을 고정하면 → 항상 같은 시드에서 시작하므로 몇 번을 실행해도 정확히 동일한 방식으로 분할됨
# test_size=0.2**는 전체 데이터 중 테스트 세트로 떼어낼 비율을 지정하는 파라미터입니다.
# 전체 데이터 중 20%는 X_test(테스트 세트)로, 나머지 80%는 X_train(학습 세트)으로 나뉩니다
# 예를 들어 이번 대화의 Santander 데이터(76,020행) 기준으로 계산하면:
# 전체: 76,020행
# 테스트 세트(20%): 약 15,204행 → X_test, y_test
# 학습 세트(80%): 약 60,816행 → X_train, y_train
X_train, X_test, y_train, y_test = train_test_split(X_features, y_labels,
                                                    test_size=0.2, random_state=0)

# y_train.count() : y_train(Series)에서 결측치(NaN)를 제외한 데이터 개수를 반환 → 학습 세트 전체 샘플 수
train_cnt = y_train.count()
# y_test.count() : y_test(Series)에서 결측치를 제외한 데이터 개수를 반환 → 테스트 세트 전체 샘플 수
test_cnt = y_test.count()

# X_train.shape, X_test.shape : 각각 (행 개수, 열 개수) 형태의 튜플 반환
# .format(0, 1) : 문자열의 {0}, {1} 자리에 순서대로 X_train.shape, X_test.shape 값을 대입
print('학습 세트 Shape:{0}, 테스트 세트 Shape:{1}'.format(X_train.shape , X_test.shape))


# 아래 출력 내용을 설명하는 라벨 출력
print(' 학습 세트 레이블 값 분포 비율')
# y_train.value_counts() : 레이블(클래스)별 등장 개수를 센 Series 반환 (내림차순 정렬)
# 여기에 train_cnt(전체 개수)로 나눠 각 클래스가 차지하는 비율(비중)로 변환
print(y_train.value_counts()/train_cnt)

# '\n' : 출력 앞에 줄바꿈을 하나 넣어 이전 출력과 구분되도록 함
print('\n 테스트 세트 레이블 값 분포 비율')
# y_test.value_counts() : 테스트 세트의 레이블별 등장 개수를 센 Series 반환
# test_cnt로 나눠 비율로 변환 (학습 세트와 분포를 비교하기 위함)
print(y_test.value_counts()/test_cnt)


학습 세트 Shape:(60816, 143), 테스트 세트 Shape:(15204, 143)
 학습 세트 레이블 값 분포 비율
TARGET
0    0.960964
1    0.039036
Name: count, dtype: float64

 테스트 세트 레이블 값 분포 비율
TARGET
0    0.9583
1    0.0417
Name: count, dtype: float64


In [ ]:
# 학습데이터를 학습데이터와 검증데이터 분리
# train_test_split() 파라미터 설명
#   - X_features : 분할할 입력(피처) 데이터
#   - y_labels   : 분할할 정답(레이블) 데이터 (X_features와 짝을 맞춰 함께 분할됨)
#   - test_size=0.2   : 전체 데이터 중 테스트 세트로 뗄 비율 (0.2 = 20%, 나머지 80%는 학습 세트)
#   - random_state=0  : 무작위 분할 시 시드값 고정. 값을 고정하면 코드를 몇 번 실행해도 동일한 결과로 분할됨
# 반환값 순서: X_train(학습용 입력), X_test(테스트용 입력), y_train(학습용 정답), y_test(테스트용 정답)

In [21]:
# X_train, y_train을 다시 학습과 검증 데이터 세트로 분리.

# train_test_split() 파라미터 설명
#   - X_train : 앞서 분리된 학습용 입력 데이터 (여기서 다시 학습/검증용으로 재분할)
#   - y_train : 앞서 분리된 학습용 정답 데이터 (X_train과 짝을 맞춰 함께 분할됨)
#   - test_size=0.3   : X_train, y_train 중 30%를 검증(validation) 세트로 분리 (나머지 70%는 최종 학습 세트)
#   - random_state=0  : 무작위 분할 시 시드값 고정, 재실행해도 동일한 결과로 분할됨
# 반환값 순서: X_tr(최종 학습용 입력), X_val(검증용 입력), y_tr(최종 학습용 정답), y_val(검증용 정답)
# → 목적: 테스트 세트(X_test, y_test)는 최종 평가용으로 남겨두고, 학습 과정 중 성능 확인(예: 조기 종료)을 위해 별도의 검증 세트를 마련
# 즉 전체 76,020행 → X_train(약 60,816행, 80%) / X_test(약 15,204행, 20%)로 이미 나뉜 상태이고, 여기서 X_train을 원재료로 삼아 또 한 번 나누는 것입니다.
# 2. test_size=0.3의 의미
# X_train(약 60,816행) 중 30%를 떼어냅니다.
# X_val(검증용): 약 60,816 × 0.3 ≈ 18,245행
# X_tr(최종 학습용): 약 60,816 × 0.7 ≈ 42,571행
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train,
                                                    test_size=0.3, random_state=0)

In [ ]:
# early_stopping_rounds=100, eval_metric="auc"
# Lgbm 사용

# lightgbm에서 분류 모델 클래스(LGBMClassifier)와 콜백 함수 early_stopping, log_evaluation 임포트
from lightgbm import LGBMClassifier, early_stopping,log_evaluation
# AUC(ROC 곡선 아래 면적) 점수를 계산하는 함수 임포트
from sklearn.metrics import roc_auc_score

# LGBMClassifier() 파라미터 설명
#   - n_estimators=500 : 생성할 트리(부스팅 반복 횟수)의 최대 개수를 500으로 설정
lgbm_clf = LGBMClassifier(n_estimators=500)

# eval_set : 학습 중 매 반복마다 성능을 평가할 (입력, 정답) 데이터셋 목록
#   - (X_tr, y_tr)   : 학습 세트 자체의 성능도 함께 모니터링
#   - (X_val, y_val) : 검증 세트 성능을 모니터링 → early stopping 판단 기준으로 사용
eval_set=[(X_tr, y_tr), (X_val, y_val)]

# lgbm_clf.fit() 파라미터 설명
#   - X_tr, y_tr           : 실제 모델 학습에 사용되는 학습 데이터
#   - eval_metric="auc"    : 학습 중 평가할 성능 지표를 AUC로 지정
#   - eval_set=eval_set    : 위에서 정의한 평가용 데이터셋 목록 전달
#   - callbacks=[...]      : 학습 중 매 반복마다 실행할 콜백 함수 목록 (LightGBM 최신 버전은 early_stopping/verbose를 콜백 방식으로 처리)
lgbm_clf.fit(
        X_tr,
        y_tr,
        eval_metric="auc", # "성능이 좋아지고 있는지"를 판단할 기준 지표를 AUC로 삼겠다는 설정
        eval_set=eval_set,
        callbacks=[
            # early_stopping(stopping_rounds=100) : 검증 세트 성능이 100번 연속 개선되지 않으면 학습을 조기 종료
            early_stopping(stopping_rounds=100),
            # log_evaluation(period=50) : 50번 반복마다 한 번씩 학습 로그(평가 결과)를 출력
            log_evaluation(period=50)
        ]
)

# lgbm_clf.predict_proba(X_test)[:, 1] : X_test에 대해 각 클래스 확률을 예측 후, 클래스 1(불만족)일 확률만 선택
predict_proba = lgbm_clf.predict_proba(X_test)[:, 1] # 1인 예측 확률,  class1
# print('y_test -> ' , y_test)
# print('predict_proba -> ' , predict_proba)

# roc_auc_score(y_test, predict_proba) : 실제 정답(y_test)과 예측 확률을 비교해 AUC 점수 계산
lgbm_roc_score = roc_auc_score(y_test, predict_proba)
# '{0:.4f}'.format(...) : 소수점 넷째 자리까지 반올림하여 LightGBM 모델의 AUC 점수 출력
# 결과 : ROC AUC: 0.8384(컬럼정제전)
# 결과 : ROC AUC: 0.8370(컬럼정제후)
print('ROC AUC: {0:.4f}'.format(lgbm_roc_score))

[LightGBM] [Info] Number of positive: 1658, number of negative: 40913
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009648 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11770
[LightGBM] [Info] Number of data points in the train set: 42571, number of used features: 143
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.038947 -> initscore=-3.205836
[LightGBM] [Info] Start training from score -3.205836
Training until validation scores don't improve for 100 rounds
[50]	training's auc: 0.914429	training's binary_logloss: 0.110108	valid_1's auc: 0.830813	valid_1's binary_logloss: 0.135596
[100]	training's auc: 0.936826	training's binary_logloss: 0.0993941	valid_1's auc: 0.829593	valid_1's binary_logloss: 0.136459
Early stopping, best iteration is:
[35]	training's auc: 0.903424	training's binary_logloss: 0.114848	valid_1's auc: 0.831735	valid_1's binary_logloss: 0.135488
ROC AUC: 0.8370


In [26]:
# hyperopt에서 하이퍼파라미터의 탐색 공간(분포/범위)을 정의하는 hp 모듈 임포트
from hyperopt import hp
# hyperopt
# 1. search space 선언

# lgbm_search_space : LightGBM 하이퍼파라미터 튜닝을 위해 hyperopt가 탐색할 파라미터 공간을 딕셔너리로 정의
#   - key   : LGBMClassifier에 실제로 전달될 파라미터 이름
#   - value : hp.함수(라벨, ...)로 정의된 탐색 범위/분포
lgbm_search_space = {
                     # hp.quniform(label, low, high, q) : low~high 범위를 q 간격의 정수형 값으로 균등 탐색
                     #   - 'num_leaves' : 트리 하나가 가질 수 있는 최대 리프(잎) 노드 개수, 모델 복잡도를 조절
                     #   - 32, 64, 1    : 32부터 64까지 1씩 증가하며 탐색
                     'num_leaves': hp.quniform('num_leaves', 32, 64, 1),

                     # hp.quniform(label, low, high, q) : 1 간격의 정수형 값 탐색
                     #   - 'max_depth'  : 트리의 최대 깊이
                     #   - 100, 160, 1  : 100부터 160까지 1씩 증가하며 탐색 (LightGBM은 -1이 깊이 제한 없음을 의미하며 num_leaves로 복잡도를 더 많이 제어하기 때문에 큰 범위도 흔히 씀)
                     'max_depth': hp.quniform('max_depth', 100, 160, 1),

                     # hp.quniform(label, low, high, q) : 1 간격의 정수형 값 탐색
                     #   - 'min_child_samples' : 리프 노드가 되기 위해 필요한 최소 데이터(샘플) 개수, 과적합 방지용
                     #   - 60, 100, 1          : 60부터 100까지 1씩 증가하며 탐색
                     'min_child_samples': hp.quniform('min_child_samples', 60, 100, 1),

                     # hp.uniform(label, low, high) : low~high 범위에서 연속적인 실수값을 균등분포로 탐색
                     #   - 'subsample' : 트리 생성 시 사용할 데이터(행) 샘플링 비율
                     #   - 0.7, 1      : 70%~100% 사이 실수값 탐색
                     'subsample': hp.uniform('subsample', 0.7, 1),

                     # hp.uniform(label, low, high) : 연속 실수값 균등분포 탐색
                     #   - 'learning_rate' : 학습률(각 트리가 예측에 반영되는 정도)
                     #   - 0.01, 0.2       : 0.01~0.2 사이 실수값 탐색
                     'learning_rate': hp.uniform('learning_rate', 0.01, 0.2)
                    }

In [29]:
# sklearn에서 K-Fold 교차검증을 위한 KFold 클래스 임포트
from sklearn.model_selection import KFold
# AUC(ROC 곡선 아래 면적) 점수를 계산하는 함수 임포트
from sklearn.metrics import roc_auc_score

# def objective_func(search_space) : hyperopt의 fmin()이 반복 호출할 목적 함수 정의
#   - search_space : fmin() 실행 시 hp로 정의된 탐색 공간에서 뽑힌 하이퍼파라미터 값들이 딕셔너리 형태로 전달됨
def objective_func(search_space):
    # LGBMClassifier() 파라미터 설명
    #   - n_estimators=100                                  : 트리 개수 100개로 고정 (튜닝 대상 아님, 속도를 위해 작게 설정)
    #   - num_leaves=int(search_space['num_leaves'])         : search_space에서 뽑힌 num_leaves 값 사용 (hp.quniform은 float 반환하므로 int 형변환)
    #   - max_depth=int(search_space['max_depth'])           : search_space에서 뽑힌 max_depth 값 사용, int 형변환
    #   - min_child_samples=int(search_space['min_child_samples']) : search_space에서 뽑힌 값 사용, int 형변환
    #   - subsample=search_space['subsample']                 : search_space에서 뽑힌 값 그대로 사용 (이미 float)
    #   - learning_rate=search_space['learning_rate']         : search_space에서 뽑힌 값 그대로 사용
    lgbm_clf =  LGBMClassifier(n_estimators=100, num_leaves=int(search_space['num_leaves']),
                               max_depth=int(search_space['max_depth']),
                               min_child_samples=int(search_space['min_child_samples']),
                               subsample=search_space['subsample'],
                               learning_rate=search_space['learning_rate'])
    # 3개 k-fold 방식으로 평가된 roc_auc 지표를 담는 list
    # roc_auc_list : 각 fold별로 계산된 AUC 점수를 저장할 빈 리스트 초기화
    roc_auc_list = []
   
    # 3개 k-fold방식 적용
    # KFold(n_splits=3) : 데이터를 3개의 폴드(그룹)로 나눠 교차검증을 수행하도록 설정하는 객체 생성
    kf = KFold(n_splits=3)
    # X_train을 다시 학습과 검증용 데이터로 분리
    # kf.split(X_train) : X_train을 3등분하여, 매 반복마다 (학습용 인덱스, 검증용 인덱스) 쌍을 하나씩 반환 → 총 3번 반복
    for tr_index, val_index in kf.split(X_train):
        # kf.split(X_train)으로 추출된 학습과 검증 index값으로 학습과 검증 데이터 세트 분리
        # X_train.iloc[tr_index], y_train.iloc[tr_index] : 이번 fold에서 학습용으로 쓸 인덱스에 해당하는 행 선택
        X_tr, y_tr = X_train.iloc[tr_index], y_train.iloc[tr_index]
        # X_train.iloc[val_index], y_train.iloc[val_index] : 이번 fold에서 검증용으로 쓸 인덱스에 해당하는 행 선택
        X_val, y_val = X_train.iloc[val_index], y_train.iloc[val_index]


        # early stopping은 30회로 설정하고 추출된 학습과 검증 데이터로 XGBClassifier 학습 수행.
        # lgbm_clf.fit() 파라미터 설명
        #   - X_tr, y_tr                        : 이번 fold의 학습용 데이터로 실제 학습 수행
        #   - callbacks=[early_stopping(30)]     : 검증 성능이 30회 연속 개선 없으면 조기 종료
        #   - eval_metric="auc"                  : 평가 지표를 AUC로 지정
        #   - eval_set=[(X_tr, y_tr), (X_val, y_val)] : 학습/검증 세트 성능을 매 반복마다 모니터링
        lgbm_clf.fit(X_tr, y_tr, callbacks=[early_stopping(30)], eval_metric="auc",
           eval_set=[(X_tr, y_tr), (X_val, y_val)])


        # 1로 예측한 확률값 추출후 roc auc 계산하고 평균 roc auc 계산을 위해 list에 결과값 담음.
        # lgbm_clf.predict_proba(X_val)[:, 1] : 검증 세트에 대해 클래스 1일 확률만 추출
        # roc_auc_score(y_val, ...) : 실제 정답(y_val)과 예측 확률을 비교해 AUC 점수 계산
        score = roc_auc_score(y_val, lgbm_clf.predict_proba(X_val)[:, 1])
        # roc_auc_list.append(score) : 이번 fold의 AUC 점수를 리스트에 추가
        roc_auc_list.append(score)
   
    # 3개 k-fold로 계산된 roc_auc값의 평균값을 반환하되,
    # HyperOpt는 목적함수의 최소값을 위한 입력값을 찾으므로 -1을 곱한 뒤 반환.
    # np.mean(roc_auc_list) : 3개 fold의 AUC 점수 평균 계산
    # -1을 곱하는 이유: hyperopt의 fmin()은 값을 "최소화"하는 방향으로 탐색하므로,
    #                  AUC는 클수록 좋은 지표이기 때문에 부호를 뒤집어 "최소화 = AUC 최대화"가 되도록 맞춤
    return -1*np.mean(roc_auc_list)

In [30]:
# hyperopt에서 최적화 실행 함수(fmin), TPE 알고리즘(tpe), 시도 기록 클래스(Trials) 임포트
from hyperopt import fmin, tpe, Trials


# Trials() : fmin() 실행 중 매 시도(trial)마다의 하이퍼파라미터 값, 점수, 상태 등을 기록/저장하는 객체 생성
#   → 나중에 전체 탐색 히스토리를 분석하거나 재사용할 때 사용
trials = Trials()


# fmin()함수를 호출. max_evals지정된 횟수만큼 반복 후 목적함수의 최소값을 가지는 최적 입력값 추출.

# fmin() 파라미터 설명
#   - fn=objective_func       : 최소화할 목적 함수 (LightGBM용, 하이퍼파라미터를 받아 -AUC 평균값을 반환)
#   - space=lgbm_search_space : 앞서 정의한 LightGBM 하이퍼파라미터 탐색 공간(딕셔너리)
#   - algo=tpe.suggest        : 탐색 알고리즘으로 TPE(Tree-structured Parzen Estimator, 베이지안 최적화 기법의 일종) 사용
#   - max_evals=50            : 목적 함수를 최대 50번 호출(50번의 하이퍼파라미터 조합 시도)하며 탐색
#   - trials=trials           : 위에서 만든 Trials 객체에 각 시도 결과를 기록
#   - rstate=np.random.default_rng(seed=30) : 탐색 과정의 무작위성을 제어하는 난수 생성기, seed 고정으로 재실행 시 동일 결과 재현
best = fmin(fn=objective_func, space=lgbm_search_space, algo=tpe.suggest,
            max_evals=50, # 최대 반복 횟수를 지정합니다.
            trials=trials, rstate=np.random.default_rng(seed=30))


# best : fmin()이 찾아낸, 목적 함수 값(-AUC 평균)을 최소로 만드는(=AUC를 최대로 만드는) 최적의 LightGBM 하이퍼파라미터 조합(딕셔너리)
print('best:', best)

[LightGBM] [Info] Number of positive: 1579, number of negative: 38965
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012947 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11720                    
[LightGBM] [Info] Number of data points in the train set: 40544, number of used features: 143
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.038945 -> initscore=-3.205872
[LightGBM] [Info] Start training from score -3.205872 
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:       
[100]	training's auc: 0.888382	training's binary_logloss: 0.121662	valid_1's auc: 0.83083	valid_1's binary_logloss: 0.135917
[LightGBM] [Info] Number of positive: 1609, number of negative: 38935
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011005 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] 

In [32]:
# LGBMClassifier() 파라미터 설명
#   - n_estimators=500                          : 튜닝 때(100)보다 트리 개수를 늘려 최종 모델의 성능을 끌어올림
#   - num_leaves=int(best['num_leaves'])         : fmin()이 찾은 최적 num_leaves 값 사용 (hp.quniform은 float 반환하므로 int 형변환)
#   - max_depth=int(best['max_depth'])           : 최적 max_depth 값 사용, int 형변환
#   - min_child_samples=int(best['min_child_samples']) : 최적 min_child_samples 값 사용, int 형변환
#   - subsample=round(best['subsample'], 5)      : 최적 subsample 값 사용, 소수점 5자리까지 반올림
#   - learning_rate=round(best['learning_rate'], 5) : 최적 learning_rate 값 사용, 소수점 5자리까지 반올림
#   - early_stopping_rounds=100                  : 검증 성능이 100회 연속 개선 없으면 조기 종료 (최종 학습이므로 튜닝 때(30)보다 넉넉하게 설정)
#   - eval_metric="auc"                          : 평가 지표를 AUC로 지정
lgbm_clf =  LGBMClassifier(n_estimators=500, num_leaves=int(best['num_leaves']),
                           max_depth=int(best['max_depth']),
                           min_child_samples=int(best['min_child_samples']),
                           subsample=round(best['subsample'], 5),
                           learning_rate=round(best['learning_rate'], 5),
                           early_stopping_rounds=100,
                           eval_metric="auc",
                          )


# evaluation metric을 auc로, early stopping은 100 으로 설정하고 학습 수행.

# lgbm_clf.fit() 파라미터 설명
#   - X_tr, y_tr    : 실제 모델 학습에 사용되는 학습 데이터 (앞서 만들어둔 학습용 세트 재사용)
#   - eval_set=[(X_tr, y_tr), (X_val, y_val)] : 학습 중 매 반복마다 학습/검증 세트 성능을 함께 모니터링 (조기 종료 판단용)
lgbm_clf.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_val, y_val)])


# roc_auc_score() 파라미터 설명
#   - y_test : 실제 정답 레이블 (최종 평가용, 학습/검증에 전혀 쓰이지 않은 데이터)
#   - lgbm_clf.predict_proba(X_test)[:,1] : X_test에 대한 클래스 1(불만족) 확률만 추출
lgbm_roc_score = roc_auc_score(y_test, lgbm_clf.predict_proba(X_test)[:,1])

# '{0:.4f}'.format(...) : 소수점 넷째 자리까지 반올림하여 최종 튜닝된 LightGBM 모델의 AUC 점수 출력
# 결과 ROC AUC --> : 0.8414(컬럼정제전)
# 결과 ROC AUC --> : 0.8417(컬럼정제후)
print('ROC AUC --> : {0:.4f}'.format(lgbm_roc_score))

[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] early_stopping_round is set=100, early_stopping_rounds=100 will be ignored. Current value: early_stopping_round=100
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Info] Number of positive: 1658, number of negative: 40913
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013928 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11770
[LightGBM] [Info] Number of data points in the train set: 42571, number of used features: 143
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] early_stopping_round is set=100, early_stopping_rounds=100 will be ignored. Current value: early_stopping_round=100
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.038947 -> initscore=-3.205836
[LightGBM] [Info] Start training from score -3.205836
Training until validation scores don't improve for 100 rounds
Early stoppin